In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error

In [2]:
# ---- Load data ----
train = pd.read_csv('../data preprocessing+EDA/train_updated.csv')
test = pd.read_csv('../data preprocessing+EDA/test_updated.csv')

In [3]:
FEATURES = [
    'Urban_rura_Urban', 
    'LU_pct_micro_2025_msoa', 
    'LU_pct_large_2025_msoa',
    'log_enterprises_per_1k_residents_2025', 
    'turnover_diversity_1-HHI_2025_msoa',
    'LU_diversity_1-HHI_2025', 
    'share_enterprises_kibs_2025_msoa',
    'emp_rate', 
    'full_time_share', 
    'log_Mid-2024 population',
    'IMD_decile'
]

TARGETS = ['log_total_GVA_2023', 'log_gva_per_worker_2023', 'log_GVA_2025_predicted']


PARAM_DIST = {
    'n_estimators'    : [100, 200, 300, 500, 700, 1000],
    'max_depth'       : [2, 3, 4, 5, 6, 7],
    'learning_rate'   : [0.005, 0.01, 0.02, 0.05, 0.1],
    'subsample'       : [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7, 10, 15],
    'reg_alpha'       : [0, 0.01, 0.05, 0.1, 0.5, 1, 2],
    'reg_lambda'      : [0.5, 1, 1.5, 2, 3, 5],
    'gamma'           : [0, 0.1, 0.2, 0.5, 1],  
}

In [ ]:
RANDOM_STATE = 42

X_train = train[FEATURES]
y_train = train[TARGETS]
X_test  = test[FEATURES]
y_test  = test[TARGETS]

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

stage1_results = {}
stage2_results = {}
fitted_models  = {}
 

In [ ]:
for target in TARGETS:
 
    X_train = train[FEATURES]
    y_train = train[target]
    X_test  = test[FEATURES]
    y_test  = test[target]
    
    # ------------------------------------------------------------
    # Stage 1: Randomized search over a wide range of hyperparameters
    # ------------------------------------------------------------

    stage1 = RandomizedSearchCV(
        xgb.XGBRegressor(objective='reg:squarederror', random_state=RANDOM_STATE, n_jobs=-1),
        param_distributions=PARAM_DIST,
        n_iter=150,       
        scoring='r2',
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    stage1.fit(X_train, y_train)
    best1 = stage1.best_params_
    y_pred = stage1.best_estimator_.predict(X_test)

    stage1_results[target] = {
            'cv_best_r2': round(stage1.best_score_, 6),
            'test_r2'   : round(r2_score(y_test, y_pred), 6),
            'test_rmse' : round(root_mean_squared_error(y_test, y_pred), 6),
            'test_mae'  : round(mean_absolute_error(y_test, y_pred), 6),
            'test_mape' : round(mean_absolute_percentage_error(y_test, y_pred) * 100, 4),
        }

    print(f"\n=== {target} | Stage 1 ===")
    print(f"Best params: {best1}")
    print(f"CV best R2:  {stage1_results[target]['cv_best_r2']}")
    print(f"Test R2:     {stage1_results[target]['test_r2']}")
    print(f"Test RMSE:   {stage1_results[target]['test_rmse']}")
    print(f"Test MAE:    {stage1_results[target]['test_mae']}")
    print(f"Test MAPE:   {stage1_results[target]['test_mape']}%")


     
    # ------------------------------------------------------------
    # Stage 2: Narrow grid search around best params
    # ------------------------------------------------------------

    PARAM_GRID = {
        'n_estimators'    : sorted(set([max(50, best1['n_estimators'] - 100),
                                        best1['n_estimators'],
                                        best1['n_estimators'] + 100])),
        'max_depth'       : sorted(set([max(2, best1['max_depth'] - 1),
                                        best1['max_depth'],
                                        min(8, best1['max_depth'] + 1)])),
        'learning_rate'   : [best1['learning_rate']],
        'subsample'       : [best1['subsample']],
        'colsample_bytree': [best1['colsample_bytree']],
        'min_child_weight': sorted(set([max(1, best1['min_child_weight'] - 2),
                                        best1['min_child_weight'],
                                        best1['min_child_weight'] + 2])),
        'reg_alpha'       : [best1['reg_alpha']],
        'reg_lambda'      : [best1['reg_lambda']],
        'gamma'           : [best1['gamma']],
    }

    stage2 = GridSearchCV(
        xgb.XGBRegressor(objective='reg:squarederror', random_state=RANDOM_STATE, n_jobs=-1),
        param_grid=PARAM_GRID,
        scoring='r2',
        cv=cv,
        n_jobs=-1
    )
    stage2.fit(X_train, y_train)
    best2  = stage2.best_params_
    y_pred = stage2.best_estimator_.predict(X_test)

    stage2_results[target] = {
        'cv_best_r2': round(stage2.best_score_, 6),
        'test_r2'   : round(r2_score(y_test, y_pred), 6),
        'test_rmse' : round(root_mean_squared_error(y_test, y_pred), 6),
        'test_mae'  : round(mean_absolute_error(y_test, y_pred), 6),
        'test_mape' : round(mean_absolute_percentage_error(y_test, y_pred) * 100, 4),
    }
    fitted_models[target] = stage2.best_estimator_

    print(f"\n=== {target} | Stage 2 ===")
    print(f"Best params: {best2}")
    print(f"CV best R2:  {stage2_results[target]['cv_best_r2']}")
    print(f"Test R2:     {stage2_results[target]['test_r2']}")
    print(f"Test RMSE:   {stage2_results[target]['test_rmse']}")
    print(f"Test MAE:    {stage2_results[target]['test_mae']}")
    print(f"Test MAPE:   {stage2_results[target]['test_mape']}%")




=== log_total_GVA_2023 | Stage 1 ===
Best params: {'subsample': 0.5, 'reg_lambda': 2, 'reg_alpha': 0.05, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.05, 'gamma': 0.5, 'colsample_bytree': 0.8}
CV best R2:  0.728193
Test R2:     0.602839
Test RMSE:   0.694876
Test MAE:    0.426714
Test MAPE:   11.7213%

=== log_total_GVA_2023 | Stage 2 ===
Best params: {'colsample_bytree': 0.8, 'gamma': 0.5, 'learning_rate': 0.05, 'max_depth': 5, 'min_child_weight': 5, 'n_estimators': 200, 'reg_alpha': 0.05, 'reg_lambda': 2, 'subsample': 0.5}
CV best R2:  0.732308
Test R2:     0.596375
Test RMSE:   0.700508
Test MAE:    0.431006
Test MAPE:   11.7798%

=== log_gva_per_worker_2023 | Stage 1 ===
Best params: {'subsample': 0.7, 'reg_lambda': 0.5, 'reg_alpha': 0, 'n_estimators': 1000, 'min_child_weight': 1, 'max_depth': 7, 'learning_rate': 0.005, 'gamma': 0.5, 'colsample_bytree': 0.7}
CV best R2:  0.298475
Test R2:     0.221822
Test RMSE:   0.680822
Test MAE:    0.414328
Te

In [9]:
s1_df = pd.DataFrame(stage1_results).T[['cv_best_r2','test_r2','test_rmse','test_mae','test_mape']]
s2_df = pd.DataFrame(stage2_results).T[['cv_best_r2','test_r2','test_rmse','test_mae','test_mape']]

print(f"\n=== Summary across all three targets | Stage 1 ===")
print(s1_df.to_string())

print(f"\n=== Summary across all three targets | Stage 2 ===")
print(s2_df.to_string())


=== Summary across all three targets | Stage 1 ===
                         cv_best_r2   test_r2  test_rmse  test_mae  test_mape
log_total_GVA_2023         0.728193  0.602839   0.694876  0.426714    11.7213
log_gva_per_worker_2023    0.298475  0.221822   0.680822  0.414328     3.6930
log_GVA_2025_predicted     0.719806  0.599528   0.713906  0.456127    12.7859

=== Summary across all three targets | Stage 2 ===
                         cv_best_r2   test_r2  test_rmse  test_mae  test_mape
log_total_GVA_2023         0.732308  0.596375   0.700508  0.431006    11.7798
log_gva_per_worker_2023    0.299073  0.222464   0.680541  0.414055     3.6896
log_GVA_2025_predicted     0.720431  0.601809   0.711870  0.453348    12.7049
